In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Bronze Layer — Metadata-Driven Ingestion
# MAGIC Ingests **one** source table into the Bronze layer. Source and target are
# MAGIC parameterized via widgets so this same notebook can be reused for every
# MAGIC table in the pipeline — an orchestrator (Databricks Workflow / job task)
# MAGIC loops over rows in a metadata/config table and calls this notebook once
# MAGIC per row, passing the values in as parameters.

# COMMAND ----------

dbutils.widgets.text("source_catalog", "main")
dbutils.widgets.text("source_schema", "raw")
dbutils.widgets.text("source_table", "customers")
dbutils.widgets.text("target_catalog", "main")
dbutils.widgets.text("target_schema", "bronze")
dbutils.widgets.text("target_table", "customers")
dbutils.widgets.text("load_type", "full")          # full | incremental
dbutils.widgets.text("watermark_column", "")        # required only for incremental

source_catalog     = dbutils.widgets.get("source_catalog")
source_schema      = dbutils.widgets.get("source_schema")
source_table       = dbutils.widgets.get("source_table")
target_catalog     = dbutils.widgets.get("target_catalog")
target_schema      = dbutils.widgets.get("target_schema")
target_table       = dbutils.widgets.get("target_table")
load_type          = dbutils.widgets.get("load_type")
watermark_column   = dbutils.widgets.get("watermark_column")

source_fqn = f"{source_catalog}.{source_schema}.{source_table}"
target_fqn = f"{target_catalog}.{target_schema}.{target_table}"

print(f"Source table : {source_fqn}")
print(f"Target table : {target_fqn}")
print(f"Load type    : {load_type}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Optional: self-lookup from a control/metadata table
# MAGIC Rather than trusting the widget values blindly, the notebook can look up
# MAGIC its own config row from a control table keyed by `target_table`. This is
# MAGIC what makes the pipeline truly "metadata-driven" rather than just
# MAGIC "parameterized" — the metadata table is the single source of truth, and
# MAGIC widgets are just how the orchestrator passes in the lookup key.

# COMMAND ----------

metadata_df = (
    spark.table("main.control.pipeline_metadata")
    .filter(f"layer = 'bronze' AND target_table = '{target_table}'")
)

if metadata_df.count() > 0:
    config = metadata_df.collect()[0].asDict()
else:
    # fall back to widget-supplied values if no metadata row exists yet
    config = {
        "source_fqn": source_fqn,
        "target_fqn": target_fqn,
        "load_type": load_type,
        "watermark_column": watermark_column,
    }

# COMMAND ----------

# MAGIC %md
# MAGIC ## Read source

# COMMAND ----------

from pyspark.sql.functions import current_timestamp, lit

df = spark.table(config.get("source_fqn", source_fqn))

if config.get("load_type", load_type) == "incremental" and config.get("watermark_column", watermark_column):
    wm_col = config.get("watermark_column", watermark_column)
    try:
        last_max = (
            spark.table(config.get("target_fqn", target_fqn))
            .agg({wm_col: "max"})
            .collect()[0][0]
        )
    except Exception:
        # target table doesn't exist yet on first run
        last_max = None

    if last_max is not None:
        df = df.filter(df[wm_col] > last_max)

df = (
    df.withColumn("_ingested_at", current_timestamp())
      .withColumn("_source_table", lit(config.get("source_fqn", source_fqn)))
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Write to Bronze

# COMMAND ----------

write_mode = "append" if config.get("load_type", load_type) == "incremental" else "overwrite"

(
    df.write
      .format("delta")
      .mode(write_mode)
      .option("mergeSchema", "true")
      .saveAsTable(config.get("target_fqn", target_fqn))
)

print(f"Bronze load complete: {config.get('target_fqn', target_fqn)} ({write_mode})")
